In [1]:
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Dashboard Donatarias", layout="wide")

# =========================================================
# CARGA DE DATOS
# =========================================================
RUTA = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"

@st.cache_data
def cargar_datos():
    generales = pd.read_excel(f"{RUTA}/Generales.xlsx")
    donativos = pd.read_excel(f"{RUTA}/Ingreso por donativos.xlsx")
    relacionados = pd.read_excel(f"{RUTA}/Ingresos relacionados.xlsx")
    no_relacionados = pd.read_excel(f"{RUTA}/Ingresos no relacionados.xlsx")
    return generales, donativos, relacionados, no_relacionados

generales, donativos, relacionados, no_relacionados = cargar_datos()

# =========================================================
# PREPARACIÓN DE VARIABLES BASE
# =========================================================

# --- Total de ingresos por organización y año (Indicador 2) ---
# Ajusta 'ID' por la llave real que identifica a cada organización (RFC, folio, etc.)
LLAVE = ["Año", "Folio"]

don_efectivo = donativos.groupby(LLAVE)["Monto efectivo"].sum().reset_index()
don_especie = donativos.groupby(LLAVE)["Monto especie"].sum().reset_index()
ing_relacionados = relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos relacionados"})
ing_no_relacionados = no_relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos no relacionados"})

total_ingresos = (
    don_efectivo
    .merge(don_especie, on=LLAVE, how="outer")
    .merge(ing_relacionados, on=LLAVE, how="outer")
    .merge(ing_no_relacionados, on=LLAVE, how="outer")
    .fillna(0)
)

total_ingresos["Total Ingresos Anio Fiscal"] = (
    total_ingresos["Monto efectivo"]
    + total_ingresos["Monto especie"]
    + total_ingresos["Ingresos relacionados"]
    + total_ingresos["Ingresos no relacionados"]
)

total_ingresos["Total Donativos Recibidos"] = (
    total_ingresos["Monto efectivo"] + total_ingresos["Monto especie"]
)

# --- Dependencia de donativos (Indicador 3) ---
total_ingresos["Dependencia de donativos"] = (
    total_ingresos["Total Donativos Recibidos"] / total_ingresos["Total Ingresos Anio Fiscal"]
).fillna(0)

# --- Quintil de ingresos (Indicador 1) ---
# Se calcula por año, ya que la distribución nacional cambia año con año
def asignar_quintil(df):
    df = df.copy()
    df["Quintil de ingresos"] = pd.qcut(
        df["Total Ingresos Anio Fiscal"].rank(method="first", ascending=False),
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    return df

total_ingresos = total_ingresos.groupby("Año", group_keys=False).apply(asignar_quintil)



2026-10-10 11:35:25.750 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 11:35:25.751 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-10 11:35:25.752 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-10 11:35:25.752 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 11:35:25.838 
  command:

    streamlit run /Users/alanrobles/Library/Python/3.9/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-10-10 11:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 11:35:25.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ig

In [ ]:
# =========================================================
# GRÁFICO 1 — Treemap de ingresos por quintil (filtrado por año)
# =========================================================
st.header("1. Monto de ingresos por quintil de ingresos")

# Excluir 2026 (sin datos aún)
total_ingresos_filtrado = total_ingresos[total_ingresos["Año"] != 2026].copy()

anio_g1 = st.selectbox(
    "Año",
    sorted(total_ingresos_filtrado["Año"].unique()),
    key="g1_anio"
)

df_g1 = total_ingresos_filtrado[total_ingresos_filtrado["Año"] == anio_g1]

monto_quintil = (
    df_g1
    .groupby("Quintil de ingresos", observed=True)["Total Ingresos Anio Fiscal"]
    .sum()
    .reindex(["Q1", "Q2", "Q3", "Q4", "Q5"])
    .reset_index()
)
monto_quintil.columns = ["Quintil", "Monto total"]

fig1 = px.treemap(
    monto_quintil,
    path=["Quintil"],
    values="Monto total",
    color="Quintil",
    title=f"Monto total de ingresos por quintil ({anio_g1})"
)
fig1.update_traces(
    texttemplate="<b>%{label}</b><br>$%{value:,.0f}",
    textfont_size=16
)
st.plotly_chart(fig1, use_container_width=True)

2026-09-12 16:02:10.665 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.666 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.666 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.675 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.676 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.677 Session state does not function when running a script without `streamlit run`
2026-09-12 16:02

DeltaGenerator()

In [ ]:
fig1

In [ ]:
# =========================================================
# GRÁFICO 2 — Total de ingresos año fiscal
# =========================================================
st.header("2. Total de ingresos año fiscal")

evolucion = total_ingresos.groupby("Año")["Total Ingresos Anio Fiscal"].sum().reset_index()

fig2 = px.line(
    evolucion, x="Año", y="Total Ingresos Anio Fiscal",
    markers=True,
    title="Evolución del total de ingresos (agregado, todas las organizaciones)"
)
fig2.update_layout(yaxis_title="Ingresos (MXN)")
st.plotly_chart(fig2, use_container_width=True)

2026-09-12 16:02:10.975 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.976 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:10.977 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:11.061 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:11.061 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:11.062 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:11.062 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 16:02:11.063 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [ ]:
fig2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# =========================================================
# CARGA DE DATOS
# =========================================================
RUTA = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"
LLAVE = ["Año", "Folio"]  # ajustar 'ID' a la llave real

generales = pd.read_excel(f"{RUTA}/Generales.xlsx")
donativos = pd.read_excel(f"{RUTA}/Ingreso por donativos.xlsx")
relacionados = pd.read_excel(f"{RUTA}/Ingresos relacionados.xlsx")
no_relacionados = pd.read_excel(f"{RUTA}/Ingresos no relacionados.xlsx")
gastos = pd.read_excel(f"{RUTA}/Gastos.xlsx")
destino = pd.read_excel(f"{RUTA}/Destino de donativos.xlsx")  # ajustar nombre de hoja si es distinto

# =========================================================
# CONSTRUCCIÓN DE VARIABLES BASE
# =========================================================
don_efectivo = donativos.groupby(LLAVE)["Monto efectivo"].sum().reset_index()
don_especie = donativos.groupby(LLAVE)["Monto especie"].sum().reset_index()
ing_relacionados = relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos relacionados"})
ing_no_relacionados = no_relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos no relacionados"})

df = (
    don_efectivo
    .merge(don_especie, on=LLAVE, how="outer")
    .merge(ing_relacionados, on=LLAVE, how="outer")
    .merge(ing_no_relacionados, on=LLAVE, how="outer")
    .fillna(0)
)

df["Total Ingresos Anio Fiscal"] = (
    df["Monto efectivo"] + df["Monto especie"] + df["Ingresos relacionados"] + df["Ingresos no relacionados"]
)
df["Total Donativos Recibidos"] = df["Monto efectivo"] + df["Monto especie"]
df["Dependencia de donativos"] = (df["Total Donativos Recibidos"] / df["Total Ingresos Anio Fiscal"]).fillna(0)

# Gastos y sustentabilidad
gastos["Total Gastos"] = (
    gastos["Monto nacional operación"] + gastos["Monto nacional admin"]
    + gastos["Monto extranjero operación"] + gastos["Monto extranjero admin"]
)
total_gastos = gastos.groupby(LLAVE)["Total Gastos"].sum().reset_index()
df = df.merge(total_gastos, on=LLAVE, how="left")
df["Total Gastos"] = df["Total Gastos"].fillna(0)
df["Sustentabilidad"] = (
    (df["Total Ingresos Anio Fiscal"] - df["Total Gastos"])
    / df["Total Ingresos Anio Fiscal"]
)

df["Sustentabilidad"] = df["Sustentabilidad"].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)
# Beneficiarios y costo por beneficiario
total_beneficiarios = destino.groupby(LLAVE)["Número de beneficiados"].sum().reset_index()
total_beneficiarios.columns = LLAVE + ["Total Beneficiarios"]
df = df.merge(total_beneficiarios, on=LLAVE, how="left")
df["Total Beneficiarios"] = df["Total Beneficiarios"].fillna(0)

df["Costo por beneficiario"] = np.where(
    df["Total Beneficiarios"] > 0,
    df["Total Ingresos Anio Fiscal"] / df["Total Beneficiarios"],
    np.nan
)

# Eliminar organizaciones con ingresos prácticamente nulos (probablemente error de captura
# o casos que de todas formas caerían en Crisálida por regla de negocio, no por cluster)
df = df[df["Total Ingresos Anio Fiscal"] >= 100]

# Winsorizar (acotar) ratios que pueden explotar cuando el denominador es muy chico
df["Sustentabilidad"] = df["Sustentabilidad"].clip(lower=-2, upper=1)

# Quintil de ingresos (por año)
def asignar_quintil(g):
    g = g.copy()
    g["Quintil de ingresos"] = pd.qcut(
        g["Total Ingresos Anio Fiscal"].rank(method="first", ascending=False), 5,
        labels=[1, 2, 3, 4, 5]
    ).astype(int)
    return g

df = df.groupby("Año", group_keys=False).apply(asignar_quintil)
df = df[df["Año"] != 2026]

# =========================================================
# PREPARAR MATRIZ DE VARIABLES PARA CLUSTERING
# =========================================================
FEATURES = [
    "Quintil de ingresos",
    "Total Ingresos Anio Fiscal",
    "Total Beneficiarios",
    "Costo por beneficiario",
    "Dependencia de donativos",
    "Sustentabilidad", # cantidad de plazas laborales ofrecidas, promedio salarios, salario minimo interior y en forntera
]

df_cluster = df.dropna(subset=["Costo por beneficiario"]).copy()

# Log-transform a las variables monetarias/de conteo (muy sesgadas)
for col in ["Total Ingresos Anio Fiscal", "Total Beneficiarios", "Costo por beneficiario"]:
    df_cluster[col + " (log)"] = np.log1p(df_cluster[col].clip(lower=0))

FEATURES_MODELO = [
    "Quintil de ingresos",
    "Total Ingresos Anio Fiscal (log)",
    "Total Beneficiarios (log)",
    "Costo por beneficiario (log)",
    "Dependencia de donativos",
    "Sustentabilidad",
]

X = df_cluster[FEATURES_MODELO].values
X_scaled = StandardScaler().fit_transform(X)

# =========================================================
# ELEGIR NÚMERO DE CLUSTERS (elbow + silhouette)
# =========================================================
inercia = []
silueta = []
rango_k = range(2, 8)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercia.append(km.inertia_)
    silueta.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(rango_k), inercia, marker="o")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("Número de clusters (k)")
axes[0].set_ylabel("Inercia")

axes[1].plot(list(rango_k), silueta, marker="o", color="darkorange")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Número de clusters (k)")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

# =========================================================
# AJUSTAR MODELO FINAL
# =========================================================
K_CLUSTERS = 4  # la definición original habla de 4 perfiles; ajusta según el codo/silhouette

kmeans_final = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=10)
df_cluster["Cluster"] = kmeans_final.fit_predict(X_scaled)

# =========================================================
# RESULTADOS PARA ETIQUETAR LOS CLUSTERS
# =========================================================
resumen_clusters = df_cluster.groupby("Cluster")[FEATURES].agg(["mean", "median", "count"])
print(resumen_clusters)

# Tamaño de cada cluster
print("\nTamaño de cada cluster:")
print(df_cluster["Cluster"].value_counts().sort_index())

/var/folders/v3/3xs7dyrd5qb1bq8wr_g9mg_m0000gq/T/ipykernel_961/3994295803.py:87: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



NameError: name 'StandardScaler' is not defined

In [ ]:
# Distribución de quintiles dentro de cada cluster (para validar contra la regla de negocio)
print("Distribución de Quintil dentro de cada cluster (%):")
print(pd.crosstab(df_cluster["Cluster"], df_cluster["Quintil de ingresos"], normalize="index").round(3) * 100)

# Percentiles globales de referencia para validar "Axolote con rendimiento de Salamandra"
p25_costo = df_cluster["Costo por beneficiario"].quantile(0.25)
p75_benef = df_cluster["Total Beneficiarios"].quantile(0.75)
print(f"\nPercentil 25 de Costo por beneficiario (global): {p25_costo:,.0f}")
print(f"Percentil 75 de Total Beneficiarios (global): {p75_benef:,.0f}")

# ¿Qué % de cada cluster cumple ambas condiciones de "alto rendimiento"?
df_cluster["Cumple_alto_rendimiento"] = (
    (df_cluster["Costo por beneficiario"] < p25_costo) & (df_cluster["Total Beneficiarios"] > p75_benef)
)
print("\n% de cada cluster que cumple criterio de alto rendimiento:")
print(df_cluster.groupby("Cluster")["Cumple_alto_rendimiento"].mean().round(3) * 100)

Distribución de Quintil dentro de cada cluster (%):


KeyError: 'Cluster'

In [ ]:
# =========================================================
# ETIQUETAR CLUSTERS SEGÚN EVIDENCIA
# =========================================================
ETIQUETAS_CLUSTER = {
    3: "Salamandra",
    2: "Axolote con rendimiento de Salamandra",
    0: "Axolote",
    1: "Crisálida",
}

df_cluster["Identidad organizacional"] = df_cluster["Cluster"].map(ETIQUETAS_CLUSTER)

# Verificación rápida
print(df_cluster["Identidad organizacional"].value_counts())

# =========================================================
# GUARDAR RESULTADO PARA USARLO EN EL STREAMLIT
# =========================================================
resultado_final = df_cluster[LLAVE + [
    "Identidad organizacional", "Cluster",
    "Quintil de ingresos", "Total Ingresos Anio Fiscal",
    "Total Beneficiarios", "Costo por beneficiario",
    "Dependencia de donativos", "Sustentabilidad"
]]

resultado_final.to_excel(f"{RUTA}/Identidad organizacional.xlsx", index=False)
print(f"\nGuardado en {RUTA}/Identidad organizacional.xlsx")

Identidad organizacional
Axolote                                  13577
Crisálida                                11592
Axolote con rendimiento de Salamandra    10072
Salamandra                                7859
Name: count, dtype: int64

Guardado en /Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet/Identidad organizacional.xlsx


In [ ]:
generales_check = pd.read_excel(f"{RUTA}/Generales.xlsx")
print(generales_check.columns.tolist())
print(generales_check[["Año", "Folio", "Rfc"]].head() if "Rfc" in generales_check.columns else "No tiene Rfc")

['Año', 'Rfc', 'Razón social', 'Folio', 'Misión', 'Valores', 'Url', 'Actividad', 'Actividades adicionales', 'Activo circulante', 'Activo fijo', 'Activo diferido', 'Total del estado de ingresos', 'Pasivo', 'Patrimonio', 'Dictamina', 'Rfc CP', 'Nombre CP', 'Número CP']
    Año             Folio           Rfc
0  2020  TRANS004248/2021  AAA040708SB8
1  2020  TRANS000059/2021  AAA060215LU2
2  2020  TRANS000362/2021  AAA060509UEA
3  2020  TRANS007590/2021  AAA061212Q15
4  2020  TRANS008743/2021  AAA061212Q15


In [ ]:
import pandas as pd

RUTA = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"

# Puente Folio -> Rfc usando Generales.xlsx
generales_check = pd.read_excel(f"{RUTA}/Generales.xlsx")
puente_folio_rfc = generales_check[["Año", "Folio", "Rfc"]].drop_duplicates()

# Reparar Identidad organizacional.xlsx agregando Rfc
identidad = pd.read_excel(f"{RUTA}/Identidad organizacional.xlsx")

print("Filas originales:", len(identidad))

identidad = identidad.merge(puente_folio_rfc, on=["Año", "Folio"], how="left")

print("Filas con Rfc asignado:", identidad["Rfc"].notna().sum())
print("Filas SIN Rfc asignado (revisar):", identidad["Rfc"].isna().sum())

identidad.to_excel(f"{RUTA}/Identidad organizacional.xlsx", index=False)
print("Guardado con Rfc agregado.")

Filas originales: 43100
Filas con Rfc asignado: 43100
Filas SIN Rfc asignado (revisar): 0
Guardado con Rfc agregado.


In [ ]:
import numpy as np
import pandas as pd

RUTA = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"

# =========================================================
# Ingresos totales (mismo cálculo que en el dashboard principal)
# =========================================================
donativos = pd.read_excel(f"{RUTA}/Ingreso por donativos.xlsx")
relacionados = pd.read_excel(f"{RUTA}/Ingresos relacionados.xlsx")
no_relacionados = pd.read_excel(f"{RUTA}/Ingresos no relacionados.xlsx")

LLAVE = ["Año", "Rfc"]

don_efectivo = donativos.groupby(LLAVE)["Monto efectivo"].sum().reset_index()
don_especie = donativos.groupby(LLAVE)["Monto especie"].sum().reset_index()
ing_relacionados = relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos relacionados"})
ing_no_relacionados = no_relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos no relacionados"})

total_ingresos = (
    don_efectivo
    .merge(don_especie, on=LLAVE, how="outer")
    .merge(ing_relacionados, on=LLAVE, how="outer")
    .merge(ing_no_relacionados, on=LLAVE, how="outer")
    .fillna(0)
)
total_ingresos["Total Ingresos Anio Fiscal"] = (
    total_ingresos["Monto efectivo"] + total_ingresos["Monto especie"]
    + total_ingresos["Ingresos relacionados"] + total_ingresos["Ingresos no relacionados"]
)

años = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

# =========================================================
# 5. Total de fuerza laboral
# =========================================================
nomina = pd.read_excel(f"{RUTA}/Nómina.xlsx")
rfc_s = nomina["Rfc"].unique()

fl = np.zeros((len(rfc_s), 2 * len(años)))

for i in range(len(rfc_s)):
    fl_i = np.zeros(len(años) * 2)
    for a in range(len(años)):
        match = nomina.loc[(nomina["Año"] == años[a]) & (nomina["Rfc"] == rfc_s[i]), ["Plantilla laboral"]]
        pl = match.iloc[0, 0] if not match.empty else None
        match = nomina.loc[(nomina["Año"] == años[a]) & (nomina["Rfc"] == rfc_s[i]), ["Voluntarios"]]
        v = match.iloc[0, 0] if not match.empty else None
        fl_i[2 * a] = v
        fl_i[2 * a + 1] = pl
    fl[i, :] = fl_i

fl_df = pd.DataFrame(
    data=fl, index=rfc_s,
    columns=[f"{tipo} {a}" for a in años for tipo in ["Voluntarios", "Empleados"]]
)

# =========================================================
# Ingresos totales por Rfc y año (df ancho, insumo de 6, 8 y 9)
# =========================================================
it = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    it_i = np.zeros(len(años))
    for a in range(len(años)):
        match = total_ingresos.loc[(total_ingresos["Año"] == años[a]) & (total_ingresos["Rfc"] == rfc_s[i]), ["Total Ingresos Anio Fiscal"]]
        it_i[a] = match.iloc[0, 0] if not match.empty else 0
    it[i, :] = it_i

it_df = pd.DataFrame(data=it, index=rfc_s, columns=años)
it_df.to_excel(f"{RUTA}/Ingresos totales.xlsx")

# Fuerza laboral total (voluntarios + empleados) por año y Rfc
fl_filtered = np.nan_to_num(fl)
fl_total = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    for a in range(len(años)):
        fl_total[i, a] = fl_filtered[i, 2 * a] + fl_filtered[i, 2 * a + 1]

fl_total_df = pd.DataFrame(data=fl_total, index=rfc_s, columns=años)
fl_df.to_excel(f"{RUTA}/flota_laboral.xlsx")
fl_total_df.to_excel(f"{RUTA}/flota_laboral_total.xlsx")
print("✅ flota_laboral_total.xlsx generado")

# =========================================================
# 6. Costo por empleado (salario promedio)
# =========================================================
ms = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    for a in range(len(años)):
        match = nomina.loc[(nomina["Año"] == años[a]) & (nomina["Rfc"] == rfc_s[i])]["Monto salarios"]
        ms[i, a] = match.iat[0] if not match.empty else np.nan

sp = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    for a in range(len(años)):
        try:
            sp[i, a] = ms[i, a] / fl[i, 2 * a + 1]
        except ZeroDivisionError:
            sp[i, a] = 0

sp_df = pd.DataFrame(data=sp, index=rfc_s, columns=años)
sp_df.to_excel(f"{RUTA}/Salario promedio.xlsx")
print("✅ Salario promedio.xlsx generado")

# =========================================================
# 7. Dependencia del voluntariado
# =========================================================
dv = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    for a in range(len(años)):
        try:
            dv[i, a] = fl[i, 2 * a] / fl_total[i, a]
        except ZeroDivisionError:
            dv[i, a] = 0

dv_df = pd.DataFrame(data=dv, index=rfc_s, columns=años)
dv_df.to_excel(f"{RUTA}/dependencia_voluntariado.xlsx")
print("✅ dependencia_voluntariado.xlsx generado")

# =========================================================
# 8. Costo del órgano de gobierno
# =========================================================
ogd = pd.read_excel(f"{RUTA}/Órgano de gobierno.xlsx")

stg = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    for a in range(len(años)):
        match = ogd.loc[(ogd["Año"] == años[a]) & (ogd["Rfc"] == rfc_s[i])]["Monto salario"]
        stg[i, a] = match.sum()

stg_df = pd.DataFrame(data=stg, index=rfc_s, columns=años)
cog_df = (stg_df / it_df).replace([np.inf, -np.inf], np.nan)
cog_df.to_excel(f"{RUTA}/costo_órgano_gubernamental.xlsx")
print("✅ costo_órgano_gubernamental.xlsx generado")

# =========================================================
# 9. Costo por beneficiario
# =========================================================
ddd = pd.read_excel(f"{RUTA}/Destino de donativos.xlsx")

ndb = np.zeros((len(rfc_s), len(años)))
for i in range(len(rfc_s)):
    for a in range(len(años)):
        match = ddd.loc[(ddd["Año"] == años[a]) & (ddd["Rfc"] == rfc_s[i])]["Número de beneficiados"]
        ndb[i, a] = match.sum()

ndb_df = pd.DataFrame(data=ndb, index=rfc_s, columns=años)
cpb_df = (it_df / ndb_df).replace([np.inf, -np.inf], np.nan)
cpb_df.to_excel(f"{RUTA}/costo por beneficiario.xlsx")
print("✅ costo por beneficiario.xlsx generado")

print("\nListo, los 5 archivos ya están en", RUTA)

✅ flota_laboral_total.xlsx generado


/var/folders/v3/3xs7dyrd5qb1bq8wr_g9mg_m0000gq/T/ipykernel_961/39659066.py:97: RuntimeWarning:

invalid value encountered in scalar divide



✅ Salario promedio.xlsx generado


/var/folders/v3/3xs7dyrd5qb1bq8wr_g9mg_m0000gq/T/ipykernel_961/39659066.py:112: RuntimeWarning:

invalid value encountered in scalar divide



✅ dependencia_voluntariado.xlsx generado
✅ costo_órgano_gubernamental.xlsx generado
✅ costo por beneficiario.xlsx generado

Listo, los 5 archivos ya están en /Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet


In [3]:
RUTA = "/Users/alanrobles/Documents/ALAN/GEOSTATS/dontarias/dataet"

@st.cache_data
def cargar_datos():
    generales = pd.read_excel(f"{RUTA}/Generales.xlsx")
    donativos = pd.read_excel(f"{RUTA}/Ingreso por donativos.xlsx")
    relacionados = pd.read_excel(f"{RUTA}/Ingresos relacionados.xlsx")
    no_relacionados = pd.read_excel(f"{RUTA}/Ingresos no relacionados.xlsx")
    gastos = pd.read_excel(f"{RUTA}/Gastos.xlsx")
    identidad = pd.read_excel(f"{RUTA}/Identidad organizacional.xlsx")
    fl_total_df = pd.read_excel(f"{RUTA}/flota_laboral_total.xlsx", index_col=0)
    sp_df = pd.read_excel(f"{RUTA}/Salario promedio.xlsx", index_col=0)
    dv_df = pd.read_excel(f"{RUTA}/dependencia_voluntariado.xlsx", index_col=0)
    cog_df = pd.read_excel(f"{RUTA}/costo_órgano_gubernamental.xlsx", index_col=0)
    cpb_df = pd.read_excel(f"{RUTA}/costo por beneficiario.xlsx", index_col=0)
    return generales, donativos, relacionados, no_relacionados, gastos, identidad, fl_total_df, sp_df, dv_df, cog_df, cpb_df

generales, donativos, relacionados, no_relacionados, gastos, identidad, fl_total_df, sp_df, dv_df, cog_df, cpb_df = cargar_datos()

LLAVE = ["Año", "Rfc"]  # ajustar 'ID' a la llave real

# PREPARACIÓN DE INDICADORES BASE 
don_efectivo = donativos.groupby(LLAVE)["Monto efectivo"].sum().reset_index()
don_especie = donativos.groupby(LLAVE)["Monto especie"].sum().reset_index()
ing_relacionados = relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos relacionados"})
ing_no_relacionados = no_relacionados.groupby(LLAVE)["Monto"].sum().reset_index().rename(columns={"Monto": "Ingresos no relacionados"})

total_ingresos = (
    don_efectivo
    .merge(don_especie, on=LLAVE, how="outer")
    .merge(ing_relacionados, on=LLAVE, how="outer")
    .merge(ing_no_relacionados, on=LLAVE, how="outer")
    .fillna(0)
)

total_ingresos["Total Ingresos Anio Fiscal"] = (
    total_ingresos["Monto efectivo"]
    + total_ingresos["Monto especie"]
    + total_ingresos["Ingresos relacionados"]
    + total_ingresos["Ingresos no relacionados"]
)
total_ingresos["Total Donativos Recibidos"] = (
    total_ingresos["Monto efectivo"] + total_ingresos["Monto especie"]
)
total_ingresos["Dependencia de donativos"] = (
    total_ingresos["Total Donativos Recibidos"] / total_ingresos["Total Ingresos Anio Fiscal"]
).fillna(0)

gastos["Total Gastos"] = (
    gastos["Monto nacional operación"] + gastos["Monto nacional admin"]
    + gastos["Monto extranjero operación"] + gastos["Monto extranjero admin"]
)
total_gastos = gastos.groupby(LLAVE)["Total Gastos"].sum().reset_index()

total_ingresos = total_ingresos.merge(total_gastos, on=LLAVE, how="left")
total_ingresos["Total Gastos"] = total_ingresos["Total Gastos"].fillna(0)

total_ingresos["Sustentabilidad"] = (
    (total_ingresos["Total Ingresos Anio Fiscal"] - total_ingresos["Total Gastos"])
    / total_ingresos["Total Ingresos Anio Fiscal"]
)

total_ingresos["Sustentabilidad"] = (
    total_ingresos["Sustentabilidad"]
    .replace([float("inf"), -float("inf")], 0)
    .fillna(0)
)

def asignar_quintil(df):
    df = df.copy()
    df["Quintil de ingresos"] = pd.qcut(
        df["Total Ingresos Anio Fiscal"].rank(method="first", ascending=False),
        5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )
    return df

total_ingresos = total_ingresos.groupby("Año", group_keys=False).apply(asignar_quintil)
total_ingresos = total_ingresos[total_ingresos["Año"] != 2026]  # sin datos aún

total_ingresos = total_ingresos.merge(
    identidad[["Año", "Rfc", "Identidad organizacional", "Total Beneficiarios", "Costo por beneficiario"]],
    on=["Año", "Rfc"], how="left"
)
total_ingresos["Identidad organizacional"] = total_ingresos["Identidad organizacional"].fillna("No clasificada")

total_ingresos = total_ingresos.merge(
    generales[["Año", "Rfc", "Razón social", "Actividad"]].drop_duplicates(subset=["Año", "Rfc"]),
    on=["Año", "Rfc"], how="left"
)

# SIDEBAR — FILTROS GLOBALES

# HELPERS DE FILTROS
def filtro_popover(label, opciones, key):
    """Filtro estándar: caso base = todas seleccionadas, clic quita una."""
    if key not in st.session_state:
        st.session_state[key] = list(opciones)

    seleccionados = st.session_state[key]
    if len(seleccionados) == len(opciones):
        resumen = "Todos"
    elif len(seleccionados) == 0:
        resumen = "Ninguno"
    else:
        resumen = f"{len(seleccionados)} de {len(opciones)}"

    with st.sidebar.popover(f"{label}: {resumen}", use_container_width=True):
        col_a, col_b = st.columns(2)
        if col_a.button("Todos", key=f"{key}_todos", use_container_width=True):
            st.session_state[key] = list(opciones)
            st.rerun()
        if col_b.button("Ninguno", key=f"{key}_ninguno", use_container_width=True):
            st.session_state[key] = []
            st.rerun()
        st.multiselect(" ", opciones, key=key, label_visibility="collapsed")

    return st.session_state[key]


def filtro_organizacion_popover(label, opciones_df, key):
    if key not in st.session_state:
        st.session_state[key] = []

    n_sel = len(st.session_state[key])
    resumen = "Todas" if n_sel == 0 else f"{n_sel} seleccionada(s)"

    with st.sidebar.popover(f"{label}: {resumen}", use_container_width=True):
        busqueda = st.text_input(
            "Buscar", key=f"{key}_busqueda",
            placeholder="Escribe el nombre...", label_visibility="collapsed"
        )

        opciones_filtradas = opciones_df
        if busqueda:
            opciones_filtradas = opciones_df[
                opciones_df["Razón social"].str.contains(busqueda, case=False, na=False)
            ]

        if n_sel > 0:
            if st.button("Limpiar selección", key=f"{key}_limpiar", use_container_width=True):
                st.session_state[key] = []
                st.rerun()

        MAX_MOSTRAR = 100
        total_resultados = len(opciones_filtradas)
        opciones_mostrar = opciones_filtradas.head(MAX_MOSTRAR)
        if total_resultados > MAX_MOSTRAR:
            st.caption(f"Mostrando {MAX_MOSTRAR} de {total_resultados} — afina la búsqueda para ver más.")
        elif total_resultados == 0:
            st.caption("Sin resultados.")

        # Contenedor con altura fija y scroll propio -> el popover deja de crecer
        # sin límite, así cabe hacia abajo y no se voltea.
        with st.container(height=280):
            for _, fila in opciones_mostrar.iterrows():
                rfc, nombre = fila["Rfc"], fila["Razón social"]
                marcado = rfc in st.session_state[key]
                nuevo = st.checkbox(nombre, value=marcado, key=f"{key}_chk_{rfc}")
                if nuevo and not marcado:
                    st.session_state[key].append(rfc)
                    st.rerun()
                elif not nuevo and marcado:
                    st.session_state[key].remove(rfc)
                    st.rerun()

    return st.session_state[key]

# =========================================================
# SIDEBAR — FILTROS GLOBALES
# =========================================================
st.sidebar.header("Filtros")

anios_disponibles = sorted(total_ingresos["Año"].unique())
anios_sel = filtro_popover("Año", anios_disponibles, key="anios_sel")

quintiles_sel = filtro_popover("Quintil de ingresos", ["Q1", "Q2", "Q3", "Q4", "Q5"], key="quintiles_sel")

identidades_disponibles = sorted(total_ingresos["Identidad organizacional"].dropna().unique())
identidad_sel = filtro_popover("Identidad organizacional", identidades_disponibles, key="identidad_sel")

actividades_disponibles = sorted(total_ingresos["Actividad"].dropna().unique())
actividad_sel = filtro_popover("Actividad", actividades_disponibles, key="actividad_sel")

orgs_disponibles = (
    total_ingresos[["Rfc", "Razón social"]]
    .dropna()
    .drop_duplicates(subset=["Rfc"])
    .sort_values("Razón social")
)
organizacion_sel = filtro_organizacion_popover("Organización", orgs_disponibles, key="organizacion_sel")

if not anios_sel:
    st.warning("Selecciona al menos un año en los filtros.")
    st.stop()

anio_sel = max(anios_sel)

df_anio = total_ingresos[total_ingresos["Año"] == anio_sel]
df_filtrado = df_anio[
    df_anio["Quintil de ingresos"].isin(quintiles_sel)
    & df_anio["Identidad organizacional"].isin(identidad_sel)
    & df_anio["Actividad"].isin(actividad_sel)
]
if organizacion_sel:
    df_filtrado = df_filtrado[df_filtrado["Rfc"].isin(organizacion_sel)]

2026-10-10 15:20:10.630 No runtime found, using MemoryCacheStorageManager
2026-10-10 15:20:10.633 No runtime found, using MemoryCacheStorageManager
2026-10-10 15:20:10.634 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 15:20:10.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 15:20:10.636 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 15:20:10.637 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 15:20:11.191 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 15:20:11.205 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-10 15:20:11.219 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored whe

In [5]:
print("Registros iniciales:", len(total_ingresos))

print("Año seleccionado:", anio_sel)
print("Registros del año:", len(df_anio))

print("Quintiles seleccionados:", quintiles_sel)
print("Identidades seleccionadas:", identidad_sel)
print("Actividades seleccionadas:", actividad_sel)

print("Registros tras los filtros:", len(df_filtrado))
print("Organizaciones seleccionadas:", organizacion_sel)

Registros iniciales: 52459
Año seleccionado: 2025
Registros del año: 8680
Quintiles seleccionados: ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
Identidades seleccionadas: ['Axolote', 'Axolote con rendimiento de Salamandra', 'Crisálida', 'No clasificada', 'Salamandra']
Actividades seleccionadas: ['Apoyo económico de donatarias autorizadas  ', 'Asistenciales', 'Becantes', 'Bibliotecas privadas con acceso al público en general', 'Culturales', 'Desarrollo social', 'Ecológicos', 'Educativa', 'Investigación científica o tecnológica ', 'Museos privados con acceso al público en general', 'Obras o servicios públicos', 'Reproducción de especies en protección y peligro de extinción']
Registros tras los filtros: 8680
Organizaciones seleccionadas: []


In [6]:
anio = 2025
g = generales[generales["Año"] == anio]
print("Filas en Generales:", len(g))
print("Rfc únicos en Generales:", g["Rfc"].nunique())
print("Razón social únicas en Generales:", g["Razón social"].nunique())
print("Folios únicos en Generales:", g["Folio"].nunique())
print("Rfc únicos en total_ingresos:", total_ingresos[total_ingresos["Año"] == anio]["Rfc"].nunique())

Filas en Generales: 10763
Rfc únicos en Generales: 10472
Razón social únicas en Generales: 10461
Folios únicos en Generales: 10763
Rfc únicos en total_ingresos: 8492
